# UFC Winner Modeling and Betting Backtest

## 1. Imports and Introduction

This notebook builds fight-prediction models based on my dataframe for each weight class, scores them with a confidence variable, and evaluates whether they can beat the sportsbooks on a rolling out-of-sample backtest.

Things to note:
- Things you find out post-fight are removed from the feature columns
- Fighter names are excluded from the features
- Weight-class-specific models are used, but no extra datasets are written to disk
- Confidence is the predicted probability assigned to the model's chosen winner.

Here, I'll be making some imports that will affect the rest of my modeling. Aside from the obvious pandas and numpy imports, I made my way over to sklearn and imported ColumnTransformer, SimpleImputer, Pipeline, OneHotEncoder, StandardScaler, and some assorted metrics. StandardScaler puts numeric data to a standard scale and OneHotEncoder makes categorical data binary. SimpleImputer makes sure that I don't have any missing columns (which I shouldn't), ColumnTransformer applies the necessary transformations for me, and pipeline runs these changes in a sequence so that things happen in order.

Brier loss and log loss are metrics that will be used to measure how my model performs, both scoring on a scale of 0 to 1 (0 being perfect). Log loss (Binary Cross-Entropy) will highly penalize my model for making very confident mistakes. Brier loss will be a bit more generous, rewarding good predictions and consistency.

Here are the classifiers I will be testing my model with:

- LogisticRegression is the most standard choice for binary classification, fitting an 'S' shaped line to the odds, returning a probability between 0 and 1. This is a must-test for any binary classification model like the one we have to predict the outcome of the fight.

- AdaBoostClassifier (Adaptive Boosting) learns by building on top of previous misclassified samples, increasing the weight of each incorrect prediction for each time through the data. The model will go over the data hundreds of times in our case to end up with a strong model that really knows the data it's given.

- GradientBoostingClassifier (Gradient Boosting) is a boosting method very similar to adaptive boosting. Instead of looking at how many times the model makes an incorrect prediction, it will determine the extent of the error using a loss function. 

- HistGradientBoostingClassifier: Just like GradientBoosting, but more useful for extremely large datasets because it bins the data. I don't have high hopes for this one, but it's worth a shot.

- XGBClassifier (Extreme Gradient Boosting) is very similar to our regular Gradient Boosting. XGBoost will additionally provide regularization to prevent overfitting. This is usually used for larger datasets, but we'll see how it goes.

- RandomForestClassifier (Random Forest) is a very well known machine learning algorithm that splits the data up into decision trees to predict an outcome. The data will be split up based on certain thresholds into hundreds of trees with each one making a prediction. The final prediction is the one that most trees agree on.

- ExtraTreesClassifier (Extra Randomized Trees) is just like Random Forest, except it splits the data totally randomly instead of finding the best spot. Sometimes more randomness is what we need!

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from pandas import MultiIndex, Int64Index
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    AdaBoostClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:0.4f}")

ROOT = Path.cwd()
if not (ROOT / "data" / "df.csv").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "df.csv"
raw_df = pd.read_csv(DATA_PATH, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

display(raw_df["WeightClass"].value_counts().rename("fights"))

C:\Users\Anthony\AppData\Local\Temp\ipykernel_13984\1700289466.py:6: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
C:\Users\Anthony\anaconda3\lib\site-packages\xgboost\compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index


Lightweight              1074
Welterweight             1026
Middleweight              783
Featherweight             754
Bantamweight              679
Light Heavyweight         501
Heavyweight               496
Flyweight                 353
Women's Strawweight       320
Women's Flyweight         239
Women's Bantamweight      213
Catch Weight               61
Women's Featherweight      29
Name: fights, dtype: int64

### What will we be doing with this data?

We are not necessarily trying to look at all of the data and choose which fighter will win, but instead looking at everything and deciding if the red corner fighter will win. The following columns will be excluded because they are not available before the fight or would create unrealistic leakage:

- RedWinner
- RedReturn
- BlueReturn
- RedFighter
- BlueFighter

The betting backtest uses expected value against the posted odds. For each fight, the model can either:
- bet red,
- bet blue,
- or pass if neither side clears the tuned EV threshold.

Return rate is the average amount returned per 1 unit staked, so values above 1.0 are profitable.

## 2. Preprocessing

In this first code cell, we will be defining a lot of the functions that will be used for this modeling stage. I'm going to break it up into a few sections to make things easy to read.

Starting from the top, we're going to be organizing our columns a little bit to get ready for modeling. These variables are stylized in all capitals so that they're easy to pick out when they're used in functions later on. We have our leakage columns defined at the top so our prediction isn't influenced at all by the outcome, as well as our identity columns to identify the fighters by name. The categorical columns and base features are what is used to train the data, so a lot of pre-fight things going in there. 

Things start getting interesting as we get to our dates, which we will doing a backtest with to determine the optimal start date of the dataset and the optimal validation year start time based on the number of fights in each division. The last of our variables shows us excluding a couple of weight classes and making a separate variable for the rest of them. We don't want to try modeling catchweight because it can be for any size of person, and the fighters likely had weird circumstances in their weight cuts. We don't want to model women's featherweight because there are hardly any fights in that division.

Below that we have our default parameters and our parameter grid with a dictionary or possible parameters to try for each estimator. Our model builders at the bottom will be a huge help when we're automatically building our pipelines over and over while testing parameters to find what we want.

In [2]:
# SECTION 1: VARIABLES & ESTIMATORS

#Making a bunch of variables to help with modeling and organize the columns
LEAKAGE_COLS = ["RedWinner", "RedReturn", "BlueReturn"]
IDENTITY_COLS = ["RedFighter", "BlueFighter"]
DIFF_FEATURE_COLS = [c for c in raw_df.columns if c.endswith("Diff")]
BASE_FEATURE_COLS = ["RedOdds", "BlueOdds", *DIFF_FEATURE_COLS, "TitleBout", "NumberOfRounds"]
RETURN_COLS = ["RedReturn", "BlueReturn", "RedOdds", "BlueOdds"]
CATEGORICAL_COLS = ['RedStance', 'BlueStance', 'Gender']
GLOBAL_START_DATES = ["2010-01-01", "2014-01-01", "2016-01-01", "2018-01-01"]
TEST_YEARS = [2021, 2022, 2023, 2024]
EXCLUDED_WEIGHT_CLASSES = {"Catch Weight", "Women's Featherweight"}
MODELED_WEIGHT_CLASSES = [wc for wc in raw_df["WeightClass"].value_counts().index if wc not in EXCLUDED_WEIGHT_CLASSES]

DEFAULT_MODEL_PARAMS = {
    "logreg": {"C": 0.5},
    "rf": {"n_estimators": 400, "min_samples_leaf": 3, "max_depth": None},
    "extra_trees": {"n_estimators": 400, "min_samples_leaf": 3, "max_depth": None},
    "gb": {"n_estimators": 250, "learning_rate": 0.04, "max_depth": 2},
    "hist_gb": {"max_depth": 5, "learning_rate": 0.03, "max_iter": 350},
    "ada": {"n_estimators": 400, "learning_rate": 0.03},
    "xgb": {"n_estimators": 300, "max_depth": 3, "learning_rate": 0.03},
}

ESTIMATOR_PARAM_GRIDS = {
    "logreg": [{"C": c} for c in [0.05, 0.10, 0.50, 1.50]],
    "rf": [
        {"n_estimators": 300, "min_samples_leaf": 2, "max_depth": None},
        {"n_estimators": 500, "min_samples_leaf": 3, "max_depth": None},
    ],
    "extra_trees": [
        {"n_estimators": 300, "min_samples_leaf": 2, "max_depth": None},
        {"n_estimators": 500, "min_samples_leaf": 3, "max_depth": None},
    ],
    "gb": [
        {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 2},
        {"n_estimators": 300, "learning_rate": 0.03, "max_depth": 2},
        {"n_estimators": 250, "learning_rate": 0.04, "max_depth": 3},
    ],
    "hist_gb": [
        {"max_depth": 3, "learning_rate": 0.05, "max_iter": 250},
        {"max_depth": 5, "learning_rate": 0.03, "max_iter": 350},
        {"max_depth": 6, "learning_rate": 0.02, "max_iter": 500},
    ],
    "ada": [
        {"n_estimators": 200, "learning_rate": 0.05},
        {"n_estimators": 400, "learning_rate": 0.03},
        {"n_estimators": 600, "learning_rate": 0.02},
    ],
    "xgb": [
        {"n_estimators": n, "max_depth": d, "learning_rate": lr, **booster}
        for booster in [{}, {"booster": "dart"}]
        for n, d, lr in [(250, 2, 0.04), (350, 3, 0.03), (300, 3, 0.03)]
    ],
}

MODEL_BUILDERS = {
    "logreg": lambda params: LogisticRegression(max_iter=3000, random_state=42, **params),
    "rf": lambda params: RandomForestClassifier(random_state=42, n_jobs=-1, **params),
    "extra_trees": lambda params: ExtraTreesClassifier(random_state=42, n_jobs=-1, **params),
    "gb": lambda params: GradientBoostingClassifier(random_state=42, **params),
    "hist_gb": lambda params: HistGradientBoostingClassifier(random_state=42, **params),
    "ada": lambda params: AdaBoostClassifier(random_state=42, **params),
    "xgb": lambda params: XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        n_jobs=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        min_child_weight=2,
        **params,
    ),
}

Our threshold grid is pivotal for the function of our bet placing process. If we start by looking down at tune_threshold, we can see that the results list is taking in a return from evaulate_betting_strategy for each threshold in the threshold grid. The evaluate_betting_strategy is looking at the threshold and the expected value (odds times predicted probability). The threshold with the highest return rate is chosen as the value best tuned for that model.

This value is highly important because without it, our model would only tell us who it thinks will win. With this, we can learn what is a good and what is a bad bet. Sportsbooks are made to make money, our model is made to make money. Betting on every fight would likely lead to the sportsbook winning, so our model needs to choose carefully which fights to bet on.

In [3]:
# SECTION 2: THRESHOLD GRID & BETTING ODDS

THRESHOLD_GRID = np.round(np.linspace(-0.02, 0.20, 45), 3)

def odds_to_decimal(odds):
    """Converts sportsbook odds to decimal odds"""
    odds = np.asarray(odds, dtype=float)
    return np.where(odds > 0, 1 + odds / 100.0, 1 + 100.0 / np.abs(odds))

def evaluate_betting_strategy(proba_red, odds_frame, threshold):
    """Scores model bets against the sportsbook odds"""
    red_dec = odds_to_decimal(odds_frame["RedOdds"])
    blue_dec = odds_to_decimal(odds_frame["BlueOdds"])
    ev_red = proba_red * red_dec - 1
    ev_blue = (1 - proba_red) * blue_dec - 1

    choose_red = ev_red >= ev_blue
    bet_mask = np.maximum(ev_red, ev_blue) > threshold
    realized_return = np.where(choose_red, odds_frame["RedReturn"].to_numpy(), odds_frame["BlueReturn"].to_numpy())
    placed_returns = realized_return[bet_mask]

    return {
        "bets": int(placed_returns.size),
        "returns": placed_returns,
        "return_rate": float(placed_returns.mean()) if placed_returns.size else np.nan,
        "profit_per_bet": float((placed_returns - 1).mean()) if placed_returns.size else np.nan,
        "threshold": float(threshold),
    }

def tune_threshold(proba_red, odds_frame, min_bets):
    """Picks the validation threshold with the best return rate"""
    results = [evaluate_betting_strategy(proba_red, odds_frame, threshold) for threshold in THRESHOLD_GRID]
    valid_results = [result for result in results if result["bets"] >= min_bets]
    best = max(valid_results or [evaluate_betting_strategy(proba_red, odds_frame, 0.0)], key=lambda x: (x["return_rate"], x["bets"]))
    return best["threshold"], best

Next, we're going to get some functions going that will preprocess our data. Prepare_feature_frame will make sure that only our feature columns remain, with decimal odds and implied probability added. I'm putting to use OneHotEncoder, SimpleImputer, StandardScaler, and Pipeline which I'd talked about at the time of the import. When these are through with the data, it will be preprocessed and ready to train.

In [4]:
# SECTION 3: PREPARING FOR PREPROCESSING

def prepare_feature_frame(frame):
    """Creates model features and sportsbook implied probability columns"""
    X = frame.reindex(columns=BASE_FEATURE_COLS).copy()

    red_dec = odds_to_decimal(X["RedOdds"])
    blue_dec = odds_to_decimal(X["BlueOdds"])
    red_imp = 1 / red_dec
    blue_imp = 1 / blue_dec

    X["RedDecimalOdds"] = red_dec
    X["BlueDecimalOdds"] = blue_dec
    X["RedImpliedProb"] = red_imp
    X["BlueImpliedProb"] = blue_imp
    return X


def make_one_hot_encoder():
    """Processing categorical data into binary with OneHotEncoder"""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(feature_cols):
    """Builds all preprocessing steps"""
    cat_cols = [c for c in CATEGORICAL_COLS if c in feature_cols]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    return ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", make_one_hot_encoder())]), cat_cols),
    ], remainder="drop")

Section 4 is rather simple: building the estimator based on a set of parameters, building the pipeline based on the estimator and preprocessor, and formatting the parameters so they're readable in a table.

In [5]:
# SECTION 4: ESTIMATOR AND PIPELINE BUILDING


def build_estimator(model_name, model_params=None):
    """Creates an unfitted estimator from the chosen model family and parameters"""
    params = {**DEFAULT_MODEL_PARAMS[model_name], **(model_params or {})}
    return MODEL_BUILDERS[model_name](params)


def build_pipeline(feature_cols, model_name, model_params=None):
    """Builds the pipeline for preprocessing and modeling"""
    return Pipeline([
        ("prep", build_preprocessor(feature_cols)),
        ("model", build_estimator(model_name, model_params)),
    ])


def format_params(model_params):
    """Formats parameters so that they can be displayed in a table"""
    if not model_params:
        return "default"
    return ", ".join(f"{key}={value}" for key, value in sorted(model_params.items()))

Here, we're going to be looking at changing certain parameters around based on the size of the data to see what works best. Since each weight class has a different number of fights, we need to use differently-sized train/test/validation splits for each dataset to make sure we're not overfitting or under-testing/validating. We also have a function that splits the data based on the year chosen, which will be useful for us testing a few different years to see what we like best. Finally, our get_backtest_data function gets the data we need based on the chosen start date and weight class. 

In [6]:
# SECTION 5: PREPARING FOR THE BACKTEST

def choose_weight_class_backtest_config(frame, weight_class):
    """Uses looser split requirements for weight classes with fewer fights"""
    total_fights = int((frame["WeightClass"] == weight_class).sum())
    for floor, config in [
        (300, {"min_train": 120, "min_val": 20, "min_test": 20, "min_val_bets": 8}),
        (150, {"min_train": 80, "min_val": 12, "min_test": 12, "min_val_bets": 5}),
        (60, {"min_train": 30, "min_val": 6, "min_test": 6, "min_val_bets": 3}),
    ]:
        if total_fights >= floor:
            return config
    return {"min_train": 15, "min_val": 3, "min_test": 3, "min_val_bets": 1}

def split_year(data, test_year):
    """Creates train, validation, and test masks for one test year"""
    val_start = pd.Timestamp(f"{test_year - 1}-01-01")
    test_start = pd.Timestamp(f"{test_year}-01-01")
    test_end = pd.Timestamp(f"{test_year + 1}-01-01")

    return (
        data["Date"] < val_start,
        (data["Date"] >= val_start) & (data["Date"] < test_start),
        (data["Date"] >= test_start) & (data["Date"] < test_end),
    )

def get_backtest_data(frame, start_date=None, weight_class=None):
    """Filters the dataframe and prepares X, y, and feature columns for a backtest"""
    data = frame.copy()
    if start_date is not None:
        data = data[data["Date"] >= pd.Timestamp(start_date)]
    if weight_class is not None:
        data = data[data["WeightClass"] == weight_class]
    data = data.sort_values("Date").reset_index(drop=True)

    X = prepare_feature_frame(data)
    y = data["RedWinner"].astype(int)
    feature_cols = [c for c in X.columns if c != "Date"]
    return data, X, y, feature_cols

Our rolling backtest is the backbone of the preprocessing stage for this project. We're able to learn so much about which parameters work best for our models and which models work the best for each weight class using this beautiful function. The rolling backtest takes in quite a few parameters, such as dataframe, parameters, start date, and weight class. The backtets builds the pipeline for each of the available 

In [7]:
# SECTION 6: THE BACKTEST

def rolling_backtest(frame, model_name, model_params=None, start_date=None, weight_class=None, min_train=500, min_val=100, min_test=100, min_val_bets=30):
    """Runs a chronological backtest for each model"""
    data, X, y, feature_cols = get_backtest_data(frame, start_date=start_date, weight_class=weight_class)
    if data.empty:
        return None

    returns, truth, proba, yearly_details = [], [], [], []

    for test_year in TEST_YEARS:
        train_mask, val_mask, test_mask = split_year(data, test_year)
        if train_mask.sum() < min_train or val_mask.sum() < min_val or test_mask.sum() < min_test:
            continue

        pipeline = build_pipeline(feature_cols, model_name, model_params=model_params)
        pipeline.fit(X.loc[train_mask, feature_cols], y.loc[train_mask])

        val_proba = pipeline.predict_proba(X.loc[val_mask, feature_cols])[:, 1]
        test_proba = pipeline.predict_proba(X.loc[test_mask, feature_cols])[:, 1]
        threshold, _ = tune_threshold(val_proba, data.loc[val_mask, RETURN_COLS], min_bets=min_val_bets)
        test_stats = evaluate_betting_strategy(test_proba, data.loc[test_mask, RETURN_COLS], threshold)

        if test_stats["bets"]:
            returns.append(test_stats["returns"])
        truth.append(y.loc[test_mask].to_numpy())
        proba.append(test_proba)
        yearly_details.append({"test_year": test_year, "threshold": threshold, "bets": test_stats["bets"], "return_rate": test_stats["return_rate"]})

    if not returns:
        return None

    flat_returns = np.concatenate(returns)
    flat_truth = np.concatenate(truth)
    flat_proba = np.concatenate(proba)

    return {
        "model": model_name,
        "model_params": model_params or {},
        "params_label": format_params(model_params or {}),
        "start_date": start_date,
        "weight_class": weight_class,
        "bets": int(flat_returns.size),
        "return_rate": float(flat_returns.mean()),
        "profit_per_bet": float((flat_returns - 1).mean()),
        "accuracy": float(accuracy_score(flat_truth, flat_proba >= 0.5)),
        "brier": float(brier_score_loss(flat_truth, flat_proba)),
        "logloss": float(log_loss(flat_truth, flat_proba, labels=[0, 1])),
        "splits": len(yearly_details),
        "yearly_details": yearly_details,
    }


def favorite_baseline(frame, weight_class=None, start_date="2021-01-01", end_date="2025-01-01"):
    """Benchmarks a plain favorite/underdog approach"""
    data = frame.copy()
    if weight_class is not None:
        data = data[data["WeightClass"] == weight_class]
    data = data[(data["Date"] >= pd.Timestamp(start_date)) & (data["Date"] < pd.Timestamp(end_date))]

    favorite_returns = np.where(data["RedOdds"] < data["BlueOdds"], data["RedReturn"], data["BlueReturn"])
    underdog_returns = np.where(data["RedOdds"] >= data["BlueOdds"], data["RedReturn"], data["BlueReturn"])

    return {
        "fights": int(len(data)),
        "favorite_return_rate": float(favorite_returns.mean()),
        "underdog_return_rate": float(underdog_returns.mean()),
    }


In [ ]:
#Runs rolling backtest function on all data start dates and estimator parameters
global_results = []
for start_date in GLOBAL_START_DATES:
    for model_name in ESTIMATOR_PARAM_GRIDS:
        result = rolling_backtest(
            raw_df,
            model_name,
            model_params=DEFAULT_MODEL_PARAMS[model_name],
            start_date=start_date,
        )
        if result is not None:
            global_results.append(result)

#Displaying our results, sorted by return rate        
global_results = pd.DataFrame(global_results).sort_values("return_rate", ascending=False).reset_index(drop=True)

display(
    global_results[
        ["model", "params_label", "start_date", "bets", "return_rate", "profit_per_bet", "accuracy", "brier", "logloss"]
    ]
)

#Displaying only results which returned a profit
display(
    global_results.query("return_rate > 1")[
        ["model", "params_label", "start_date", "bets", "return_rate", "profit_per_bet", "accuracy", "brier"]
    ]
)


In [9]:
all_weight_classes = MODELED_WEIGHT_CLASSES

excluded_weight_classes = sorted(EXCLUDED_WEIGHT_CLASSES)

#Running through the rolling backtest with each of our weight classes to see which estimator best fits each one
weight_results = []
for weight_class in all_weight_classes:
    class_config = choose_weight_class_backtest_config(raw_df, weight_class)
    fight_count = int((raw_df["WeightClass"] == weight_class).sum())

    for model_name, param_grid in ESTIMATOR_PARAM_GRIDS.items():
        for model_params in param_grid:
            result = rolling_backtest(
                raw_df,
                model_name,
                model_params=model_params,
                weight_class=weight_class,
                **class_config,
            )
            if result is not None:
                result["fight_count"] = fight_count
                result["config_label"] = format_params(class_config)
                weight_results.append(result)

#Organizing our results
weight_results = pd.DataFrame(weight_results)

weight_results = weight_results.sort_values(
    ["weight_class", "return_rate", "bets", "accuracy", "brier"],
    ascending=[True, False, False, False, True],
).reset_index(drop=True)

#Shows where weight classes are profitable with more than 10 bets
profitability_bet_floor = 10
print(f"Excluded weight classes that will use __global__: {excluded_weight_classes}")
profitable_weight_results = weight_results.query("return_rate > 1 and bets >= @profitability_bet_floor").copy()

display(
    weight_results[
        [
            "weight_class",
            "fight_count",
            "model",
            "params_label",
            "splits",
            "bets",
            "return_rate",
            "profit_per_bet",
            "accuracy",
            "brier",
            "logloss",
        ]
    ]
)

display(
    profitable_weight_results[
        [
            "weight_class",
            "fight_count",
            "model",
            "params_label",
            "splits",
            "bets",
            "return_rate",
            "profit_per_bet",
            "accuracy",
            "brier",
        ]
    ]
)


KeyboardInterrupt: 

In [6]:
# Pick one tuned model per modeled division, excluding catch weight bouts and women's featherweight bouts.
# Those will route to the __global__ fallback instead. 
# Sorted by return rate, number of bets, and so on
best_weight_class_models = (
    weight_results.assign(is_profitable=weight_results["return_rate"] > 1)
    .sort_values(
        ["weight_class", "is_profitable", "return_rate", "bets", "accuracy", "brier", "splits"],
        ascending=[True, False, False, False, False, True, False],
    )
    .groupby("weight_class", as_index=False)
    .head(1)
    .sort_values("weight_class")
    .reset_index(drop=True)
)

#Getting together all of our best models
final_specs = best_weight_class_models[
    ["weight_class", "model", "model_params", "params_label", "fight_count", "splits", "bets", "return_rate", "profit_per_bet", "accuracy", "brier", "logloss"]
].copy()

# Organizing all of our best models and showing how they perform against the books baseline favorites
baseline_rows = []
for weight_class in final_specs["weight_class"]:
    baseline = favorite_baseline(raw_df, weight_class=weight_class)
    baseline_rows.append({"weight_class": weight_class, **baseline})

baseline_df = pd.DataFrame(baseline_rows)
final_summary = final_specs.merge(baseline_df, on="weight_class", how="left")

display(
    final_summary[
        [
            "weight_class",
            "fight_count",
            "model",
            "params_label",
            "splits",
            "bets",
            "return_rate",
            "profit_per_bet",
            "accuracy",
            "brier",
            "favorite_return_rate",
            "underdog_return_rate",
        ]
    ].sort_values(["return_rate", "bets"], ascending=[False, False])
)


,weight_class,fight_count,model,params_label,splits,bets,return_rate,profit_per_bet,accuracy,brier,favorite_return_rate,underdog_return_rate
1,Featherweight,754,gb,"learning_rate=0.03, max_depth=2, n_estimators=300",4,95,1.2529,0.2529,0.6096,0.2215,0.8933,1.0231
4,Light Heavyweight,501,rf,"max_depth=None, min_samples_leaf=3, n_estimato...",4,66,1.2313,0.2313,0.6045,0.2242,0.9583,0.9318
3,Heavyweight,496,logreg,C=0.05,4,60,1.2299,0.2299,0.7447,0.1821,1.0715,0.7009
6,Middleweight,783,logreg,C=0.05,4,72,1.1822,0.1822,0.6485,0.2145,1.0158,0.8684
10,Women's Strawweight,320,logreg,C=0.5,4,97,1.1798,0.1798,0.6350,0.2417,0.9533,0.9937
5,Lightweight,1074,logreg,C=0.5,4,94,1.1352,0.1352,0.7045,0.2122,0.9969,0.9245
9,Women's Flyweight,239,ada,"learning_rate=0.03, n_estimators=400",3,78,1.1238,0.1238,0.5851,0.2486,0.9579,0.9633
7,Welterweight,1026,xgb,"learning_rate=0.03, max_depth=3, n_estimators=300",4,158,1.0897,0.0897,0.6777,0.2049,0.9972,0.8169
8,Women's Bantamweight,213,gb,"learning_rate=0.04, max_depth=3, n_estimators=250",4,65,1.0787,0.0787,0.5915,0.2896,0.9830,0.9472
0,Bantamweight,679,logreg,C=0.1,4,151,1.0691,0.0691,0.7100,0.2068,1.0286,0.7899


In [7]:
#Getting the best validation year for each division
def choose_deployment_validation_year(data, preferred_year=2024):
    """Pick the latest year that still leaves earlier fights available for training and enough fights for threshold tuning.
    Returns an integer year or `None` if the division is too thin for a held-out tuning year."""
    available_years = sorted(data["Date"].dt.year.dropna().unique().tolist(), reverse=True)
    ordered_years = []
    if preferred_year in available_years:
        ordered_years.append(preferred_year)
    ordered_years.extend(year for year in available_years if year != preferred_year)

    for year in ordered_years:
        year_start = pd.Timestamp(f"{year}-01-01")
        year_end = pd.Timestamp(f"{year + 1}-01-01")
        train_count = int((data["Date"] < year_start).sum())
        valid_count = int(((data["Date"] >= year_start) & (data["Date"] < year_end)).sum())
        min_valid = max(1, min(8, valid_count))
        if train_count >= max(10, min(50, len(data) // 4)) and valid_count >= min_valid:
            return int(year)
    return None

#Fitting the model
def fit_deployment_model(frame, model_name, model_params=None, weight_class=None, start_date=None, validation_year=2024):
    data = frame.copy()
    #Fitting to the preferred start date
    if start_date is not None:
        data = data[data["Date"] >= pd.Timestamp(start_date)]
    #Choosing the proper weight class
    if weight_class is not None:
        data = data[data["WeightClass"] == weight_class]
    data = data.sort_values("Date").reset_index(drop=True)

    #Getting implied odds from sportsbook
    X = prepare_feature_frame(data)
    y = data["RedWinner"].astype(int)
    feature_cols = [c for c in X.columns if c != "Date"]
    
    #Choosing the validation year and EV threshold
    chosen_validation_year = choose_deployment_validation_year(data, preferred_year=validation_year)
    if chosen_validation_year is None:
        threshold = 0.0
    else:
        valid_start = pd.Timestamp(f"{chosen_validation_year}-01-01")
        valid_end = pd.Timestamp(f"{chosen_validation_year + 1}-01-01")
        tune_train_mask = data["Date"] < valid_start
        tune_valid_mask = (data["Date"] >= valid_start) & (data["Date"] < valid_end)
        
        tuning_pipeline = build_pipeline(feature_cols, model_name, model_params=model_params)
        tuning_pipeline.fit(X.loc[tune_train_mask, feature_cols], y.loc[tune_train_mask])
        valid_proba = tuning_pipeline.predict_proba(X.loc[tune_valid_mask, feature_cols])[:, 1]
        min_bets = max(1, min(10, int(np.ceil(tune_valid_mask.sum() / 3))))
        threshold, _ = tune_threshold(valid_proba, data.loc[tune_valid_mask, RETURN_COLS], min_bets=min_bets)

    #Building and fitting the pipeline
    final_pipeline = build_pipeline(feature_cols, model_name, model_params=model_params)
    final_pipeline.fit(X.loc[:, feature_cols], y)

    return {
        "model": final_pipeline,
        "model_name": model_name,
        "model_params": model_params or {},
        "params_label": format_params(model_params or {}),
        "weight_class": weight_class,
        "start_date": start_date,
        "feature_columns": feature_cols,
        "threshold": float(threshold),
        "validation_year_used": chosen_validation_year,
    }


deployment_registry = {}
deployment_rows = []

#Keeping track of which model we're using for which weight class
for row in final_specs.to_dict("records"):
    fitted = fit_deployment_model(
        raw_df,
        row["model"],
        model_params=row["model_params"],
        weight_class=row["weight_class"],
    )
    deployment_registry[row["weight_class"]] = fitted
    deployment_rows.append(
        {
            "segment": row["weight_class"],
            "model": row["model"],
            "params_label": row["params_label"],
            "tuning_year": fitted["validation_year_used"],
            "deployment_threshold": fitted["threshold"],
        }
    )

# Global fallback for catchweight fights, women's featherweight bouts, and other fights with less data
global_fallback = fit_deployment_model(
    raw_df,
    "logreg",
    model_params=DEFAULT_MODEL_PARAMS["logreg"],
    start_date="2016-01-01",
)
deployment_registry["__global__"] = global_fallback
deployment_rows.append(
    {
        "segment": "__global__",
        "model": "logreg",
        "params_label": format_params(DEFAULT_MODEL_PARAMS["logreg"]),
        "tuning_year": global_fallback["validation_year_used"],
        "deployment_threshold": global_fallback["threshold"],
    }
)

display(pd.DataFrame(deployment_rows))



def predict_fight(fight_row, registry=deployment_registry):
    #Copying the input to a dataframe, extra instances there for error prevention
    if isinstance(fight_row, dict):
        fight_df = pd.DataFrame([fight_row])
    elif isinstance(fight_row, pd.Series):
        fight_df = fight_row.to_frame().T
    else:
        fight_df = fight_row.copy()

    if len(fight_df) != 1:
        raise ValueError("predict_fight expects exactly one fight row.")

    #Getting the weight class
    weight_class = fight_df.iloc[0]["WeightClass"]
    #Getting our model and parameters, returning global if that weight class doesn't exist
    bundle = registry.get(weight_class, registry["__global__"])
    #Preparing our features for predicting
    feature_frame = prepare_feature_frame(fight_df)
    X_pred = feature_frame.reindex(columns=bundle["feature_columns"], fill_value=np.nan)

    #Getting probability for both red and blue
    prob_red = float(bundle["model"].predict_proba(X_pred)[0, 1])
    prob_blue = 1 - prob_red
    red_name = fight_df.iloc[0].get("RedFighter", "Red")
    blue_name = fight_df.iloc[0].get("BlueFighter", "Blue")

    #Showing our model's predicted winner
    predicted_corner = "Red" if prob_red >= prob_blue else "Blue"
    predicted_winner = red_name if predicted_corner == "Red" else blue_name
    confidence = prob_red if predicted_corner == "Red" else prob_blue

    #Deciding whether or not to bet on the fight based on the probability vs the return.
    red_decimal = float(odds_to_decimal([fight_df.iloc[0]["RedOdds"]])[0])
    blue_decimal = float(odds_to_decimal([fight_df.iloc[0]["BlueOdds"]])[0])
    ev_red = prob_red * red_decimal - 1
    ev_blue = prob_blue * blue_decimal - 1

    if ev_red >= ev_blue:
        bet_corner = "Red"
        bet_name = red_name
        best_ev = ev_red
    else:
        bet_corner = "Blue"
        bet_name = blue_name
        best_ev = ev_blue

    recommended_bet = bet_name if best_ev > bundle["threshold"] else "Pass"

    return pd.Series(
        {
            "weight_class_model_used": bundle["weight_class"] or "GLOBAL_FALLBACK",
            "estimator": bundle["model_name"],
            "model_params": bundle["params_label"],
            "predicted_winner": predicted_winner,
            "confidence": confidence,
            "prob_red": prob_red,
            "prob_blue": prob_blue,
            "expected_value_red": ev_red,
            "expected_value_blue": ev_blue,
            "bet_threshold": bundle["threshold"],
            "recommended_bet": recommended_bet,
            "recommended_bet_corner": bet_corner if recommended_bet != "Pass" else "Pass",
        }
    )

,segment,model,params_label,tuning_year,deployment_threshold
0,Bantamweight,logreg,C=0.1,2024,-0.0150
1,Featherweight,gb,"learning_rate=0.03, max_depth=2, n_estimators=300",2024,0.0100
2,Flyweight,hist_gb,"learning_rate=0.02, max_depth=6, max_iter=500",2024,-0.0200
3,Heavyweight,logreg,C=0.05,2024,-0.0200
4,Light Heavyweight,rf,"max_depth=None, min_samples_leaf=3, n_estimato...",2024,0.1600
5,Lightweight,logreg,C=0.5,2024,0.0450
6,Middleweight,logreg,C=0.05,2024,-0.0200
7,Welterweight,xgb,"learning_rate=0.03, max_depth=3, n_estimators=300",2024,0.1950
8,Women's Bantamweight,gb,"learning_rate=0.04, max_depth=3, n_estimators=250",2024,0.1800
9,Women's Flyweight,ada,"learning_rate=0.03, n_estimators=400",2024,0.1450


In [8]:
testweight_results = []

testclass_config = choose_weight_class_backtest_config(raw_df, "Middleweight")
testfight_count = int((raw_df["WeightClass"] == "Middleweight").sum())

test = rolling_backtest(
    raw_df,
    "xgb",
    model_params={"booster": 'dart',"n_estimators": 300, "max_depth": 3, "learning_rate": 0.03},
    weight_class="Middleweight",
    **testclass_config,
)

if test is not None:
    test["fight_count"] = testfight_count
    test["config_label"] = format_params(testclass_config)
    testweight_results.append(test)

testweight_results = pd.DataFrame(testweight_results)
    
testweight_results = testweight_results.sort_values(
    ["weight_class", "return_rate", "bets", "accuracy", "brier"],
    ascending=[True, False, False, False, True],
).reset_index(drop=True)
    
display(
    testweight_results[
        [
            "weight_class",
            "fight_count",
            "model",
            "params_label",
            "splits",
            "bets",
            "return_rate",
            "profit_per_bet",
            "accuracy",
            "brier",
            "logloss",
        ]
    ]
)

,weight_class,fight_count,model,params_label,splits,bets,return_rate,profit_per_bet,accuracy,brier,logloss
0,Middleweight,783,xgb,"booster=dart, learning_rate=0.03, max_depth=3,...",4,151,0.9597,-0.0403,0.6151,0.2323,0.6614


In [9]:
pd.set_option("display.max_rows", 200)

weight_results

display(
    weight_results[
        [
            "weight_class",
            "fight_count",
            "model",
            "params_label",
            "splits",
            "bets",
            "return_rate",
            "profit_per_bet",
            "accuracy",
            "brier",
            "logloss",
        ]
    ]
)

,weight_class,fight_count,model,params_label,splits,bets,return_rate,profit_per_bet,accuracy,brier,logloss
0,Bantamweight,679,logreg,C=0.1,4,151,1.0691,0.0691,0.7100,0.2068,0.6143
1,Bantamweight,679,logreg,C=0.05,4,120,1.0156,0.0156,0.7013,0.2044,0.6036
2,Bantamweight,679,hist_gb,"learning_rate=0.02, max_depth=6, max_iter=500",4,172,0.9339,-0.0661,0.6623,0.2342,0.7143
3,Bantamweight,679,xgb,"booster=dart, learning_rate=0.04, max_depth=2,...",4,126,0.9306,-0.0694,0.6580,0.2116,0.6124
4,Bantamweight,679,xgb,"learning_rate=0.04, max_depth=2, n_estimators=250",4,129,0.9279,-0.0721,0.6710,0.2098,0.6068
...,...,...,...,...,...,...,...,...,...,...,...
248,Women's Strawweight,320,gb,"learning_rate=0.03, max_depth=2, n_estimators=300",4,95,0.8645,-0.1355,0.6058,0.2706,0.7709
249,Women's Strawweight,320,extra_trees,"max_depth=None, min_samples_leaf=2, n_estimato...",4,102,0.8575,-0.1425,0.5839,0.2516,0.7222
250,Women's Strawweight,320,hist_gb,"learning_rate=0.03, max_depth=5, max_iter=350",4,117,0.8419,-0.1581,0.6058,0.3001,0.8988
251,Women's Strawweight,320,rf,"max_depth=None, min_samples_leaf=2, n_estimato...",4,117,0.7943,-0.2057,0.5547,0.2564,0.7360


In [10]:
weight_results = weight_results[weight_results['weight_class'] == 'Middleweight']

## Using the Model

1. Build a one-row DataFrame that matches the pre-fight schema from df.csv
2. Call predict_fight(your_row)
3. Read predicted_winner for the straight winner pick and confidence for the 0-1 confidence score
4. If you care about betting value instead of just winner prediction, use recommended_bet and the expected value columns
5. Catch Weight and Women's Featherweight are intentionally excluded from the per-division model search and will use the __global__ fallback, since those divisions are less popular

In [11]:
upcoming_fight = pd.DataFrame([
    {
        "RedFighter": "Jiri Prochazka",
        "BlueFighter": "Carlos Ulberg",
        "RedOdds": -125,
        "BlueOdds": 105,
        "OddsDiff": -230,
        "AgeDiff": -2,
        "ReachDiff": 7.78,
        "HeightDiff": -2.54,
        "WinsDiff": -4,
        "LossesDiff": 1,
        "RoundsDiff": 1,
        "WinStreakDiff": -7,
        "LoseStreakDiff": 0,
        "RankDiff": 1,
        "WeightClass": "Light Heavyweight",
        "Gender": "MALE",
        "TitleBout": True,
        "NumberOfRounds": 5,
        "BlueCurrentLoseStreak": 0,
        "BlueCurrentWinStreak": 9,
        "BlueLongestWinStreak": 9,
        "BlueLosses": 1,
        "BlueTotalRoundsFought": 20,
        "BlueTotalTitleBouts": 0,
        "BlueWinsByKO": 6,
        "BlueWinsBySubmission": 1,
        "BlueWins": 10,
        "BlueStance": "Orthodox",
        "BlueHeightCms": 190.5,
        "BlueReachCms": 195.58,
        "BlueAge": 35,
        "BMatchWCRank": 3,
        "RedCurrentLoseStreak": 0,
        "RedCurrentWinStreak": 2,
        "RedLongestWinStreak": 3,
        "RedLosses": 2,
        "RedTotalRoundsFought": 21,
        "RedTotalTitleBouts": 3,
        "RedWinsByKO": 5,
        "RedWinsBySubmission": 1,
        "RedWins": 6,
        "RedStance": "Orthodox",
        "RedHeightCms": 187.96,
        "RedReachCms": 203.2,
        "RedAge": 33,
        "RMatchWCRank": 2,
    }
])

predict_fight(upcoming_fight)

weight_class_model_used                                    Light Heavyweight
estimator                                                                 rf
model_params               max_depth=None, min_samples_leaf=3, n_estimato...
predicted_winner                                              Jiri Prochazka
confidence                                                            0.5325
prob_red                                                              0.5325
prob_blue                                                             0.4675
expected_value_red                                                   -0.0415
expected_value_blue                                                  -0.0416
bet_threshold                                                         0.1600
recommended_bet                                                         Pass
recommended_bet_corner                                                  Pass
dtype: object